### Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

import torch
import torchinfo
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

from pathlib import Path
Root = Path('.').absolute().parent
# DATA = Root/ r'C:\Users\Admin\Projects\ML Projects\ManipDetect\data'
DATA = Root/ r'C:\Users\krishnadas\Projects\ML Projects\ManipDetect\data'

In [2]:
def prepare_lstm_sequences(df, sequence:str, sequence_name:str, sequence_length:int):
    """Prepare sequences for LSTM training
    Choose the correct df based on the sequence type
    sequence: 'daily', 'hourly', or 'weekly'
    sequence_name: 'date', 'hour', or 'week'
    """
    assert sequence in ['daily', 'hourly', 'weekly'], \
        "Invalid sequence type. Choose from 'daily', 'hourly', or 'weekly'."
    assert sequence_name in ['date', 'hour', 'week'], \
        "Invalid sequence name. Choose from 'date', 'hour', or 'week'."
    assert sequence_length in [7, 24, 4], \
        "Invalid sequence length. Choose from 7 (daily), 24 (hourly), or 4 (weekly)."
    try:
        # df = label_manipulation_periods_sequence(df, sequence, sequence_name)
        df = df.sort_values(sequence_name).reset_index(drop=True)
        # Select feature columns (exclude date and target)
        feature_cols = [col for col in df.columns 
                        if col not in [sequence_name, 'is_manipulation','date', 'week']]
        # Create sequences
        X, y = [], []
        
        for i in range(sequence_length, len(df)):
            # Use previous 7 days to predict current day
            X.append(df[feature_cols].iloc[i-sequence_length:i].values)
            y.append(df['is_manipulation'].iloc[i])
        
    except Exception as e:
        print(f"Error preparing sequences: {e}")
        return None, None, None
    
    return np.array(X), np.array(y), feature_cols

In [3]:
# load the data
hourly_data = pd.read_pickle(DATA/'hourly_data.pkl')
daily_data = pd.read_pickle(DATA/'daily_data.pkl')
weekly_data = pd.read_pickle(DATA/'weekly_data.pkl')
wsb_data = pd.read_pickle(DATA/'df_wsb_data.pkl')

X_hourly, y_hourly, feature_cols_hourly = prepare_lstm_sequences(hourly_data, sequence='hourly', sequence_name='hour', sequence_length=24)
X_daily, y_daily, feature_cols_daily = prepare_lstm_sequences(daily_data, sequence='daily', sequence_name='date', sequence_length=7)
X_weekly, y_weekly, feature_cols_weekly = prepare_lstm_sequences(weekly_data, sequence='weekly', sequence_name='week', sequence_length=4)

In [4]:
# tranform the data to tensor format
hourly_tensor = torch.FloatTensor(X_hourly)
daily_tensor = torch.FloatTensor(X_daily)
weekly_tensor = torch.FloatTensor(X_weekly)

label_tensor = torch.FloatTensor(y_weekly)

In [5]:
print(hourly_tensor.shape)
print(daily_tensor.shape)
print(weekly_tensor.shape)


torch.Size([5898, 24, 17])
torch.Size([357, 7, 9])
torch.Size([49, 4, 4])


In [6]:
hourly_tensor

tensor([[[-3.2000e-02, -5.0000e-01,  8.0000e-01,  ...,  5.0000e+00,
           6.6667e-01,  3.3333e-01],
         [ 1.3100e-01, -1.6390e-01,  6.5000e-01,  ...,  5.0000e+00,
           8.3333e-01,  1.6667e-01],
         [-2.1800e-02, -6.5500e-02,  0.0000e+00,  ...,  3.0000e+00,
           7.5000e-01,  2.5000e-01],
         ...,
         [ 4.0000e-01,  4.0000e-01,  4.0000e-01,  ...,  1.0000e+00,
           5.0000e-01,  5.0000e-01],
         [ 1.7500e-01,  0.0000e+00,  3.5000e-01,  ...,  2.0000e+00,
           6.6667e-01,  3.3333e-01],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  2.0000e+00,
           6.6667e-01,  3.3333e-01]],

        [[ 1.3100e-01, -1.6390e-01,  6.5000e-01,  ...,  5.0000e+00,
           8.3333e-01,  1.6667e-01],
         [-2.1800e-02, -6.5500e-02,  0.0000e+00,  ...,  3.0000e+00,
           7.5000e-01,  2.5000e-01],
         [ 2.3650e-01, -7.5000e-02,  1.0000e+00,  ...,  5.0000e+00,
           6.6667e-01,  3.3333e-01],
         ...,
         [ 1.7500e-01,  0

### Hierarchical Dataset

In [7]:
class HierarchicalLSTMDataset(Dataset):
    def __init__(self, hourly_data, daily_data, weekly_data, labels):
        self.hourly_data = torch.FloatTensor(hourly_data)
        self.daily_data = torch.FloatTensor(daily_data)
        self.weekly_data = torch.FloatTensor(weekly_data)
        self.labels = labels # labels correspond only to weekly_data

    def __len__(self):
        return len(self.original_data)
    
    def __getitem__(self, idx):
        return {
            'hourly': self.hourly_data[idx],
            'daily': self.daily_data[idx], 
            'weekly': self.weekly_data[idx],
            'label': self.labels[idx]
        }

### Hourly LSTM

In [8]:
class HourlyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, self.hidden_size, self.num_layers, batch_first=True)

        # Attention mechanism to focus on most suspicious hours
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )
        # Output projection to create hourly embedding
        self.output_projection = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size = x.size(0)
        # initialize the hidden state
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)

        #LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x, (h0,c0))

        # Apply attention to focus on suspicious time periods
        attention_out, attention_weights = self.attention(lstm_out,lstm_out,lstm_out)

        # Global max pooling to capture strongest coordination signals
        pooled = torch.max(attention_out, dim=1)[0]  # (batch_size, hidden_size)
        
        # Project to embedding space
        hourly_embedding = self.output_projection(pooled)
        hourly_embedding = self.dropout(hourly_embedding)
        
        return hourly_embedding, attention_weights

In [9]:
# Create dummy hourly_data
batch_size = 10
hourly_features_size = 17
seq_len = 24

model = HourlyLSTM(input_size=hourly_features_size)
dummy_input = torch.randn(batch_size, seq_len, hourly_features_size)
hourly_embedding, attention_weights = model(dummy_input)
# Print the output shapes
print(f"Input shape: {dummy_input.shape}")
print(f"Hourly embedding shape: {hourly_embedding.shape}")  # Expected: (batch_size, hidden_size // 2)
print(f"Attention weights shape: {attention_weights.shape}")  # Expected: (batch_size, seq_len, seq_len)

torchinfo.summary(model, input_size=(batch_size, seq_len, hourly_features_size))

Input shape: torch.Size([10, 24, 17])
Hourly embedding shape: torch.Size([10, 32])
Attention weights shape: torch.Size([10, 24, 24])


Layer (type:depth-idx)                   Output Shape              Param #
HourlyLSTM                               [10, 32]                  --
├─LSTM: 1-1                              [10, 24, 64]              54,528
├─MultiheadAttention: 1-2                [10, 24, 64]              16,640
├─Linear: 1-3                            [10, 32]                  2,080
├─Dropout: 1-4                           [10, 32]                  --
Total params: 73,248
Trainable params: 73,248
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 13.11
Input size (MB): 0.02
Forward/backward pass size (MB): 0.13
Params size (MB): 0.23
Estimated Total Size (MB): 0.37

### Daily LSTM

In [10]:
class DailyLSTM(nn.Module):
    def __init__(self, daily_input_size, hourly_embedding_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        combined_input_size = daily_input_size + hourly_embedding_size
        # fusion layer to combine the daily features and hourly embeddings
        self.fusion_layer = nn.Sequential(nn.Linear(combined_input_size , combined_input_size),
                                        nn.ReLU(),
                                        nn.Dropout(dropout))
        self.lstm = nn.LSTM(combined_input_size , self.hidden_size, self.num_layers, batch_first=True)
        
        # Output projection
        self.output_projection = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, daily_features, hourly_embeddings):
        """
        daily_features: (batch_size, 7, daily_input_size)
        hourly_embeddings: (batch_size, 7, hourly_embedding_size)  
        Returns: (batch_size, hidden_size//2) - weekly embedding
        """
        batch_size = daily_features.size(0)
        
        # Combine daily features with hourly embeddings
        combined_input = torch.cat([daily_features, hourly_embeddings], dim=-1)
        
        # Apply fusion layer with residual connection
        fused_input = self.fusion_layer(combined_input)
        fused_input = fused_input + combined_input  # Residual connection
        
        # Initialize hidden states
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(daily_features.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(daily_features.device)
        
        # LSTM forward pass
        lstm_out, _ = self.lstm(fused_input, (h0, c0))
        
        # Take the last output (most recent day pattern)
        daily_embedding = self.output_projection(lstm_out[:, -1, :])
        daily_embedding = self.dropout(daily_embedding)
        
        return daily_embedding


In [11]:
batch_size = 1000
seq_len = 7
input_size = 17
hourly_embedding_size = 32
model = DailyLSTM(daily_input_size=input_size, hourly_embedding_size=hourly_embedding_size)

# Simulate input data
# Simulate input data for daily features and hourly embeddings
daily_features_shape = (batch_size, seq_len, input_size)
hourly_embeddings_shape = (batch_size, seq_len, hourly_embedding_size)

# Pass the inputs to the model
torchinfo.summary(model, input_size=(daily_features_shape, hourly_embeddings_shape))

Layer (type:depth-idx)                   Output Shape              Param #
DailyLSTM                                [1000, 32]                --
├─Sequential: 1-1                        [1000, 7, 49]             --
│    └─Linear: 2-1                       [1000, 7, 49]             2,450
│    └─ReLU: 2-2                         [1000, 7, 49]             --
│    └─Dropout: 2-3                      [1000, 7, 49]             --
├─LSTM: 1-2                              [1000, 7, 64]             62,720
├─Linear: 1-3                            [1000, 32]                2,080
├─Dropout: 1-4                           [1000, 32]                --
Total params: 67,250
Trainable params: 67,250
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 443.57
Input size (MB): 1.37
Forward/backward pass size (MB): 6.58
Params size (MB): 0.27
Estimated Total Size (MB): 8.23

### Weekly LSTM

In [12]:
class WeeklyLSTM(nn.Module):
    def __init__(self, weekly_input_size, daily_embedding_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # combined input size
        combined_input_size = weekly_input_size + daily_embedding_size

        # LSTM for processing weekly sequences
        self.lstm = nn.LSTM(input_size=combined_input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=dropout if num_layers>1 else 0,
                            batch_first=True)
        
        # Classification head
        self.classifier = nn.Sequential(nn.Linear(hidden_size, hidden_size//2),
                                        nn.ReLU(),
                                        nn.Dropout(dropout),
                                        nn.Linear(hidden_size//2, hidden_size//4),
                                        nn.ReLU(),
                                        nn.Dropout(dropout),
                                        nn.Linear(hidden_size//4, 1),
                                        nn.Sigmoid())
        
    def forward(self, weekly_features, daily_embeddings):
        """
        weekly_features: (batch_size, 4, weekly_input_size)
        daily_embeddings: (batch_size, 4, daily_embedding_size)
        Returns: (batch_size, 1) - manipulation probability
        """
        batch_size = weekly_features.size(0)

        # combine weekly features with daily embeddings
        combined_input = torch.cat([weekly_features, daily_embeddings], dim=-1)

        # initialize hidden state
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)

        # LSTM forward pass
        lstm_out,_ = self.lstm(combined_input, (h0,c0))

        # Take the last output for final prediction
        final_output = lstm_out[:, -1, :]

        # Classification
        manipulation_prob = self.classifier(final_output)

        return manipulation_prob


In [13]:
# Use correct input sizes based on your data
weekly_input_size = 4  # matches feature_cols_weekly
daily_embedding_size = 32  # should match output of DailyLSTM
seq_len = 4
batch_size = 10
model = WeeklyLSTM(weekly_input_size=weekly_input_size, daily_embedding_size=daily_embedding_size)
weekly_feature_shape = (batch_size, seq_len, weekly_input_size)
daily_embedding_shape = (batch_size, seq_len, daily_embedding_size)

torchinfo.summary(model, input_size=(weekly_feature_shape, daily_embedding_shape))

Layer (type:depth-idx)                   Output Shape              Param #
WeeklyLSTM                               [10, 1]                   --
├─LSTM: 1-1                              [10, 4, 64]               59,392
├─Sequential: 1-2                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 32]                  2,080
│    └─ReLU: 2-2                         [10, 32]                  --
│    └─Dropout: 2-3                      [10, 32]                  --
│    └─Linear: 2-4                       [10, 16]                  528
│    └─ReLU: 2-5                         [10, 16]                  --
│    └─Dropout: 2-6                      [10, 16]                  --
│    └─Linear: 2-7                       [10, 1]                   17
│    └─Sigmoid: 2-8                      [10, 1]                   --
Total params: 62,017
Trainable params: 62,017
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 2.40
Input size (MB): 0.01
Forward/backwar

In [14]:
# from torchviz import make_dot
# from IPython.display import Image, display

# model = WeeklyLSTM(weekly_input_size=10, daily_embedding_size=5)
# weekly_features = torch.randn(32, 4, 10)
# daily_embeddings = torch.randn(32, 4, 5)
# output = model(weekly_features, daily_embeddings)
# dot = make_dot(output, params=dict(model.named_parameters()))
# output_path = "WeeklyLSTM"
# dot.render(output_path, format="png")

# # Display the image in the notebook
# display(Image(filename=output_path + ".png"))

### Hierarchical LSTM

In [ ]:
class HierarchicalManipulationDetector(nn.Module):
    """
    Complete hierarchical LSTM model for market manipulation detection
    """
    def __init__(self, hourly_features, daily_features, weekly_features, hidden_size=64, dropout=0.2):
        super(HierarchicalManipulationDetector, self).__init__()
        
        # Initialize the three hierarchical levels
        self.hourly_lstm = HourlyLSTM(
            input_size=hourly_features,
            hidden_size=hidden_size,
            dropout=dropout
        )
        
        self.daily_lstm = DailyLSTM(
            daily_input_size=daily_features,
            hourly_embedding_size=hidden_size // 2,
            hidden_size=hidden_size,
            dropout=dropout
        )
        
        self.weekly_lstm = WeeklyLSTM(
            weekly_input_size=weekly_features,
            daily_embedding_size=hidden_size // 2,
            hidden_size=hidden_size,
            dropout=dropout
        )

    def forward(self, hourly_data, daily_data, weekly_data):
        """
        Forward pass through all three hierarchical levels
        
        hourly_data: (batch_size, 24, hourly_features) - 7 days × 24 hours
        daily_data: (batch_size, 7, daily_features) - 7 days  
        weekly_data: (batch_size, 4, weekly_features) - 4 weeks
        
        Returns:
        - manipulation_probability: (batch_size, 1)
        - attention_weights: List of attention weights for interpretability
        """
        batch_size = hourly_data.size(0)
        
        hourly_emb = []
        attention_weights_list = []
        
        # Reshape hourly data to process each day separately
        # hourly_reshaped = hourly_data.view(batch_size * 7, 24, hourly_data.size(-1))
        
        # Process through hourly LSTM to get a single embedding for the time period
        hourly_emb, attention_weights = self.hourly_lstm(hourly_data)
        
        # Expand hourly embedding to match daily sequence length
        # Repeat the hourly embedding for each day in the sequence
        hourly_emb = hourly_emb.unsqueeze(1).repeat(1, daily_data.size(1), 1)
        # hourly_emb shape: (batch_size, 7, hidden_size//2)
        
        attention_weights_list.append(attention_weights)
        
        # Process daily data with hourly embeddings
        # Both daily_data and hourly_emb should now be (batch_size, 7, features)
        daily_embeddings = []
        
        # Generate daily embeddings (we'll create 4 of them for the 4 weeks)
        for week_idx in range(4):  # 4 weeks
            # For each week, use the same daily pattern
            # In practice, you'd want different daily patterns for different weeks
            
            daily_emb = self.daily_lstm(daily_data, hourly_emb)
            daily_embeddings.append(daily_emb)
        
        # Stack daily embeddings: (batch_size, 4, daily_embedding_size)
        daily_embeddings = torch.stack(daily_embeddings, dim=1)
        
        # Final prediction through weekly LSTM
        manipulation_probability = self.weekly_lstm(weekly_data, daily_embeddings)
        
        return manipulation_probability, attention_weights_list

In [32]:
import torch

batch_size = 10
hourly_features = 17
daily_features = 9
weekly_features = 4

model = HierarchicalManipulationDetector(
    hourly_features=hourly_features,
    daily_features=daily_features,
    weekly_features=weekly_features
)

# Correct input dimensions:
# hourly_data: 7 days × 24 hours = 168 time steps
example_hourly = torch.randn(batch_size, 24, hourly_features)  # 24 hours
# daily_data: 7 days worth of daily features
example_daily = torch.randn(batch_size, 7, daily_features)      # 7 days
# weekly_data: 4 weeks worth of weekly features  
example_weekly = torch.randn(batch_size, 4, weekly_features)    # 4 weeks

print(f"Input shapes:")
print(f"Hourly data: {example_hourly.shape}")
print(f"Daily data: {example_daily.shape}")
print(f"Weekly data: {example_weekly.shape}")

torchinfo.summary(model, input_data=(example_hourly, example_daily, example_weekly))

Input shapes:
Hourly data: torch.Size([10, 24, 17])
Daily data: torch.Size([10, 7, 9])
Weekly data: torch.Size([10, 4, 4])


Layer (type:depth-idx)                   Output Shape              Param #
HierarchicalManipulationDetector         [10, 1]                   --
├─HourlyLSTM: 1-1                        [10, 32]                  --
│    └─LSTM: 2-1                         [10, 24, 64]              54,528
│    └─MultiheadAttention: 2-2           [10, 24, 64]              16,640
│    └─Linear: 2-3                       [10, 32]                  2,080
│    └─Dropout: 2-4                      [10, 32]                  --
├─DailyLSTM: 1-2                         [10, 32]                  --
│    └─Sequential: 2-5                   [10, 7, 41]               --
│    │    └─Linear: 3-1                  [10, 7, 41]               1,722
│    │    └─ReLU: 3-2                    [10, 7, 41]               --
│    │    └─Dropout: 3-3                 [10, 7, 41]               --
│    └─LSTM: 2-6                         [10, 7, 64]               60,672
│    └─Linear: 2-7                       [10, 32]                  

In [26]:
daily_tensor.size(1)

7

In [36]:
# Create dummy hourly_data
batch_size = 10
weekly_features = 9
daily_data = torch.randn(batch_size, 7, weekly_features)
print(daily_data)

tensor([[[-1.4642e+00, -8.0803e-01,  1.2871e-01,  4.1248e-01,  2.1352e-01,
           2.6484e-01,  1.8079e+00, -1.9777e-01, -7.6337e-01],
         [ 7.6592e-01,  1.4632e-01, -6.5169e-01,  3.4402e-01, -2.4563e-01,
           7.2448e-02,  1.0885e-01, -7.7543e-02,  7.1240e-01],
         [-7.3368e-01,  7.5798e-01,  6.0691e-01,  8.1476e-01, -1.6105e-01,
           1.5402e-01, -8.7258e-01, -2.6947e-01,  1.0030e+00],
         [-7.6139e-01, -6.5371e-02, -2.0847e-02,  4.2677e-02, -5.9214e-01,
           2.3677e-01,  2.2309e+00, -4.6559e-01,  1.9490e+00],
         [ 2.3717e+00,  4.6234e-01, -1.1360e+00, -6.7226e-01,  4.5363e-01,
          -1.5648e+00, -2.4321e-01, -1.7069e+00, -4.1168e-01],
         [ 5.5099e-01,  1.1322e+00, -2.2869e-01,  3.1872e-01, -1.8183e+00,
           3.5638e-01,  7.4548e-02,  2.4782e+00, -8.3228e-01],
         [-1.3247e+00,  8.9757e-01, -1.7911e+00, -1.4961e+00, -3.0739e-01,
           1.2043e+00, -4.1228e-01,  3.5379e-01,  1.2593e+00]],

        [[ 8.5455e-01,  2.6278e-

In [42]:
week_idx = 0
week_daily_data = daily_data[:, week_idx*7:(week_idx+1)*7, :] if week_idx < 4 else daily_data[:, -7:, :]

print(f"Input Shape: {daily_data.shape}")  # Expected: (32, 168, 10)
print(f"Day {week_idx} Shape: {week_daily_data.shape}")

Input Shape: torch.Size([10, 7, 9])
Day 0 Shape: torch.Size([10, 7, 9])


In [75]:
hourly_lstm = HourlyLSTM(17)
hourly_emb, attention_weights = hourly_lstm(day_hourly_data)
print(hourly_emb.shape)
print(attention_weights.shape)

torch.Size([10, 32])
torch.Size([10, 24, 24])


In [27]:
class ManipulationDetectorTrainer:
    """Training class for the hierarchical model"""
    
    def __init__(self, model, device='cpu', learning_rate=0.001, class_weights=None):
        self.model = model.to(device)
        self.device = device
        
        # Handle class imbalance with weighted loss
        if class_weights is not None:
            pos_weight = torch.tensor([class_weights[1] / class_weights[0]]).to(device)
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        else:
            self.criterion = nn.BCELoss()
            
        self.optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=10, gamma=0.5)
        
        # Training history
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        
    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0
        correct_predictions = 0
        total_samples = 0
        
        for batch in dataloader:
            hourly_data = batch['hourly'].to(self.device)
            daily_data = batch['daily'].to(self.device)
            weekly_data = batch['weekly'].to(self.device)
            labels = batch['label'].to(self.device)
            
            self.optimizer.zero_grad()
            
            # Forward pass
            predictions, _ = self.model(hourly_data, daily_data, weekly_data)
            predictions = predictions.squeeze()
            
            # Calculate loss
            loss = self.criterion(predictions, labels)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            
            # Calculate accuracy
            predicted_labels = (predictions > 0.5).float()
            correct_predictions += (predicted_labels == labels).sum().item()
            total_samples += labels.size(0)
        
        avg_loss = total_loss / len(dataloader)
        accuracy = correct_predictions / total_samples
        
        return avg_loss, accuracy
    
    def evaluate(self, dataloader):
        self.model.eval()
        total_loss = 0
        correct_predictions = 0
        total_samples = 0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for batch in dataloader:
                hourly_data = batch['hourly'].to(self.device)
                daily_data = batch['daily'].to(self.device)
                weekly_data = batch['weekly'].to(self.device)
                labels = batch['label'].to(self.device)
                
                predictions, _ = self.model(hourly_data, daily_data, weekly_data)
                predictions = predictions.squeeze()
                
                loss = self.criterion(predictions, labels)
                total_loss += loss.item()
                
                predicted_labels = (predictions > 0.5).float()
                correct_predictions += (predicted_labels == labels).sum().item()
                total_samples += labels.size(0)
                
                all_predictions.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        avg_loss = total_loss / len(dataloader)
        accuracy = correct_predictions / total_samples
        
        return avg_loss, accuracy, all_predictions, all_labels
    
    def train(self, train_loader, val_loader, epochs=50, patience=10):
        """Complete training loop with early stopping"""
        
        print("Training Hierarchical LSTM...")
        print("="*50)
        
        best_val_accuracy = 0
        patience_counter = 0
        start_time = time.time()
        
        for epoch in range(epochs):
            # Training
            train_loss, train_acc = self.train_epoch(train_loader)
            
            # Validation
            val_loss, val_acc, _, _ = self.evaluate(val_loader)
            
            # Learning rate scheduling
            self.scheduler.step()
            
            # Store history
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            
            print(f'Epoch {epoch+1}/{epochs}:')
            print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
            print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
            print(f'  LR: {self.optimizer.param_groups[0]["lr"]:.2e}')
            
            # Early stopping
            if val_acc > best_val_accuracy:
                best_val_accuracy = val_acc
                patience_counter = 0
                torch.save(self.model.state_dict(), 'best_hierarchical_lstm.pth')
                print(f'  ✓ New best model saved! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
                
            if patience_counter >= patience:
                print(f'\nEarly stopping after {epoch+1} epochs')
                break
            
            print()
        
        training_time = time.time() - start_time
        print(f"Training completed in {training_time:.2f} seconds")
        print(f"Best validation accuracy: {best_val_accuracy:.4f}")
        
        return best_val_accuracy
    
    def plot_training_history(self):
        """Plot training curves"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss curves
        ax1.plot(self.train_losses, label='Train Loss', color='blue')
        ax1.plot(self.val_losses, label='Val Loss', color='red')
        ax1.set_title('Training and Validation Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Accuracy curves
        ax2.plot(self.train_accuracies, label='Train Acc', color='blue')
        ax2.plot(self.val_accuracies, label='Val Acc', color='red')
        ax2.set_title('Training and Validation Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def analyze_performance(self, test_loader, plot_roc=True):
        """Comprehensive performance analysis"""
        print("\n" + "="*50)
        print("HIERARCHICAL LSTM PERFORMANCE ANALYSIS")
        print("="*50)
        
        test_loss, test_acc, predictions, labels = self.evaluate(test_loader)
        
        print(f"Test Accuracy: {test_acc:.4f}")
        print(f"Test Loss: {test_loss:.4f}")
        
        # Convert to numpy
        predictions = np.array(predictions)
        labels = np.array(labels)
        predicted_labels = (predictions > 0.5).astype(int)
        
        # AUC Score
        try:
            auc_score = roc_auc_score(labels, predictions)
            print(f"AUC Score: {auc_score:.4f}")
        except:
            print("AUC Score: Could not calculate (single class in test set)")
        
        # Classification report
        print("\nClassification Report:")
        print(classification_report(labels, predicted_labels, 
                                  target_names=['Normal', 'Manipulation']))
        
        # Confusion matrix
        cm = confusion_matrix(labels, predicted_labels)
        print(f"\nConfusion Matrix:")
        print(f"                Predicted")
        print(f"Actual    Normal  Manipulation")
        print(f"Normal      {cm[0,0]:4d}      {cm[0,1]:4d}")
        print(f"Manip       {cm[1,0]:4d}      {cm[1,1]:4d}")
        
        return test_acc, predictions, labels

In [ ]:
# Example usage and demonstration
if __name__ == "__main__":
    print("SIMPLIFIED HIERARCHICAL LSTM ARCHITECTURE")
    print("="*50)
    
    # Model parameters (based on your preprocessing pipeline)
    hourly_features = 17  # post_volume, sentiment, text_similarity, etc.
    daily_features = 9   # aggregated daily patterns
    weekly_features = 4   # long-term trends
    hidden_size = 64
    
    print(f"Input dimensions:")
    print(f"  Hourly features: {hourly_features}")
    print(f"  Daily features: {daily_features}")
    print(f"  Weekly features: {weekly_features}")
    print(f"  Hidden size: {hidden_size}")

    # Create datasets
    train_dataset = HierarchicalLSTMDataset(train_hourly, train_daily, train_weekly, train_labels)
    test_dataset = HierarchicalLSTMDataset(test_hourly, test_daily, test_weekly, test_labels)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    # Initialize model
    model = HierarchicalManipulationDetector(
        hourly_features=hourly_features,
        daily_features=daily_features,
        weekly_features=weekly_features,
        hidden_size=hidden_size,
        dropout=0.2
    )
    
    print(f"\nModel Architecture:")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    trainer = ManipulationDetectorTrainer(model, device='cuda' if torch.cuda.is_available() else 'cpu')
    best_acc = trainer.train(train_loader, val_loader, epochs=50)
    trainer.plot_training_history()
    trainer.analyze_performance(test_loader)

SIMPLIFIED HIERARCHICAL LSTM ARCHITECTURE
Input dimensions:
  Hourly features: 15
  Daily features: 12
  Weekly features: 8
  Hidden size: 64

Model Architecture:
Total parameters: 201,277


NameError: name 'train_loader' is not defined